# Lightweight ML for IoT Intrusion Detection — V3
## Compatible with pandas 3.1.1 + numpy 2.5.4 + Python 3.12

**V3 changes vs V2:**
- 100% pandas 3.x / numpy 2.x compatible (no `inplace=True`, no deprecated options)
- **Priority 1**: multi-seed stability experiments (10 seeds, mean ± std)
- **Priority 2**: cross-dataset validation supports TON_IoT, BoT_IoT, and Edge-IIoTset
- **Priority 3**: hard-attack deep dive (Scan + MITM feature analysis)
- **Priority 4**: noise-robustness methodology fixed (raw-feature noise + per-feature relative noise)

---

## 0 · Setup

In [ ]:
# =====================================================================
# SECTION 0 - SETUP  (pandas 3.1.1 / numpy 2.5.4 compatible)
# =====================================================================
import os, sys, gc, time, random, hashlib, platform, warnings, shutil, zipfile
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, matthews_corrcoef,
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, brier_score_loss, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve,
)
from sklearn.exceptions import ConvergenceWarning

SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
# NOTE: mode.use_inf_as_na was removed in pandas 2.2+; do not call it.

# ------------------------- run configuration -------------------------
FAST_MODE = bool(int(os.environ.get("FAST_MODE", "0")))
TEST_FRAC, VAL_FRAC = 0.25, 0.15
SHAP_SAMPLE    = 500   if FAST_MODE else 2000
LATENCY_N      = 80    if FAST_MODE else 300
BATCH_N        = 10_000
FEATURE_KS     = [5, 10, 20, 76] if FAST_MODE else [5, 10, 15, 20, 30, 45, 76]
# FIXED noise levels - much smaller, more realistic (Priority 4)
NOISE_SIGMAS   = [0.0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5] if not FAST_MODE else [0.0, 0.05, 0.2]
DROPOUT_PCTS   = [0, 20] if FAST_MODE else [0, 10, 20, 30]
LABEL_NOISE_RATES = [0.0, 0.05, 0.10, 0.20]
FGSM_EPSILONS  = [0.0, 0.01, 0.05, 0.1, 0.2]
PGD_STEPS      = 5
BOOTSTRAP_N    = 200 if FAST_MODE else 500
# Priority 1: multi-seed experiment
MULTI_SEEDS    = [42, 123, 456, 789, 2024, 314, 271, 828, 919, 1000] if not FAST_MODE else [42, 123, 456]
COST_FN, COST_FP = 10.0, 1.0

RUN_CROSS_DATASET = bool(int(os.environ.get("RUN_CROSS_DATASET", "1")))

DATA_DIR, FIG_DIR, RESULTS_DIR, MODELS_DIR = (
    Path("data"), Path("figures"), Path("results"), Path("models")
)
for _d in (DATA_DIR, FIG_DIR, RESULTS_DIR, MODELS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 300, "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "legend.frameon": False, "axes.axisbelow": True,
})
MODEL_COLORS = {"Logistic Regression": "#0072B2", "XGBoost": "#D55E00", "Neural Network": "#009E73"}
CAT_COLORS = {"Normal": "#009E73", "Mirai": "#D55E00", "DoS": "#E69F00",
              "Scan": "#0072B2", "MITM ARP Spoofing": "#CC79A7"}

RESULTS_ROWS, PROBS, PREDS, EFFICIENCY = [], {}, {}, {}

def banner(title):
    print("\n" + "=" * 76 + f"\n  {title}\n" + "=" * 76)

def finish_fig(fig, name):
    out = FIG_DIR / f"{name}.png"
    fig.savefig(out, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f"[figure saved] {out}")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
print(f"setup OK | FAST_MODE = {FAST_MODE}")
print(f"python = {sys.version.split()[0]} | numpy = {np.__version__} | pandas = {pd.__version__}")


## 0b · Environment report

In [ ]:
import importlib.metadata as _md
import psutil
def _pv(pkg):
    try: return _md.version(pkg)
    except Exception: return "not installed"
env_info = {
    "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "python": sys.version.split()[0], "platform": platform.platform(),
    "cpu_logical": os.cpu_count(),
    "ram_GB": round(psutil.virtual_memory().total / 1e9, 1),
    "fast_mode": FAST_MODE, "seed": SEED,
    "numpy": _pv("numpy"), "pandas": _pv("pandas"), "scikit-learn": _pv("scikit-learn"),
    "xgboost": _pv("xgboost"), "torch": _pv("torch"), "shap": _pv("shap"),
    "matplotlib": _pv("matplotlib"), "seaborn": _pv("seaborn"),
}
pd.Series(env_info, name="value").to_csv(RESULTS_DIR / "environment.csv")
banner("ENVIRONMENT")
for _k, _v in env_info.items():
    print(f"  {_k:<15} {_v}")


## 1 · Data acquisition

**IoTID20 (required):** https://www.kaggle.com/datasets/rohulaminlabid/iotid20-dataset
Place `IoT Network Intrusion Dataset.csv` inside `data/`.

**Cross-dataset (optional, Priority 2):** place any of these inside `data/`:
- TON_IoT: https://www.kaggle.com/datasets/gwdaraktle/toniot-datasets
- BoT_IoT: https://www.kaggle.com/datasets/aldostronsky/botiot
- Edge-IIoTset: https://www.kaggle.com/datasets/mohamedaminebouzid/edge-iiotset-iotiot-cyber-security-dataset

The notebook auto-detects whichever is present.

In [ ]:
KAGGLE_SLUG = "rohulaminlabid/iotid20-dataset"
LEAK_COLS = ["Flow_ID", "Src_IP", "Dst_IP", "Src_Port", "Dst_Port", "Timestamp", "Sub_Cat"]

def find_dataset_parts(base=DATA_DIR):
    hits = []
    for pat in ("*.csv", "*.csv.gz", "*.zip"):
        for p in Path(base).rglob(pat):
            nl = p.name.lower()
            if "iot network intrusion" in nl or "iotid20" in nl:
                hits.append(p)
    csvs = sorted(p for p in hits if p.suffix.lower() == ".csv")
    gzs  = sorted(p for p in hits if p.name.lower().endswith(".csv.gz"))
    zips = sorted(p for p in hits if p.suffix.lower() == ".zip")
    return csvs or gzs or zips

def _slim(df):
    for c in df.select_dtypes(include="float64").columns:
        df = df.astype({c: "float32"})
    return df

def load_parts(paths):
    frames, ts_parts, cat_parts = [], [], []
    for k, p in enumerate(paths):
        print(f"  reading [{k+1}/{len(paths)}] {p.name} ...")
        df = pd.read_csv(p, low_memory=False)
        ts_parts.append(df["Timestamp"].astype(str) if "Timestamp" in df.columns else None)
        cat_parts.append(df["Cat"].astype(str).str.strip() if "Cat" in df.columns else None)
        df = df.drop(columns=[c for c in LEAK_COLS if c in df.columns])
        frames.append(_slim(df))
    ts_raw  = pd.concat(ts_parts,  ignore_index=True) if ts_parts and ts_parts[0]  is not None else None
    cat_raw = pd.concat(cat_parts, ignore_index=True) if cat_parts and cat_parts[0] is not None else None
    raw = frames[0] if len(frames) == 1 else pd.concat(frames, ignore_index=True)
    del frames; gc.collect()
    return raw, ts_raw, cat_raw

csv_paths = find_dataset_parts()
if not csv_paths:
    raise FileNotFoundError(
        "No IoTID20 file found under ./data.\n"
        "Download from https://www.kaggle.com/datasets/rohulaminlabid/iotid20-dataset\n"
        "and place 'IoT Network Intrusion Dataset.csv' inside ./data/"
    )
print(f"{len(csv_paths)} dataset file(s) found")
raw, ts_raw, cat_raw = load_parts(csv_paths)
print(f"loaded: {raw.shape[0]:,} rows x {raw.shape[1]} feature cols")


## 2A · Load, clean, align

In [ ]:
banner("SECTION 2A - LOAD & CLEAN")
_t0 = time.time()
n_rows_raw = len(raw)
print(f"raw shape: {raw.shape} | memory: {raw.memory_usage(deep=True).sum()/1e6:,.0f} MB")

missing_pct = (raw.isna().mean() * 100).sort_values(ascending=False)
num_cols_raw = raw.select_dtypes(include=[np.number]).columns
def _safe_inf_count(df, cols):
    total = 0
    for c in cols:
        try:
            arr = df[c].to_numpy(dtype='float64', na_value=np.nan)
            total += int(np.isinf(arr).sum())
        except Exception:
            pass
    return total
n_inf = _safe_inf_count(raw, num_cols_raw)
n_sentinel = int(sum((raw[c] == -1).sum() for c in num_cols_raw))
label_uniques = sorted(raw["Label"].astype(str).str.strip().unique()) if "Label" in raw.columns else []
print(f"missing: {int(raw.isna().sum().sum()):,} | inf: {n_inf:,} | -1 sentinels: {n_sentinel:,}")
print(f"Label values: {label_uniques}")

data = raw.copy()
del raw; gc.collect()

num_cols = data.select_dtypes(include=[np.number]).columns.tolist()
# Replace inf with NaN (pandas 3.x safe - no inplace)
data[num_cols] = data[num_cols].replace([np.inf, -np.inf], np.nan)
for c in num_cols:
    data.loc[data[c] == -1, c] = np.nan

# Coerce object columns
obj_cols = [c for c in data.columns if data[c].dtype == object and c not in ("Label", "Cat", "y_cat")]
for c in obj_cols:
    conv = pd.to_numeric(data[c], errors="coerce")
    if conv.notna().mean() >= 0.5:
        data[c] = conv
    else:
        data = data.drop(columns=c)

data["_ts"] = pd.to_datetime(ts_raw, errors="coerce", dayfirst=True) if ts_raw is not None else pd.NaT
data["y_cat"] = cat_raw if cat_raw is not None else "Unknown"
data["y_bin"] = (data["Label"].astype(str).str.strip() == "Anomaly").astype(np.int64)
data = data.drop(columns=["Label"] + (["Cat"] if "Cat" in data.columns else []))

FEATURE_COLS = [c for c in data.columns if c not in ("y_bin", "y_cat", "_ts")]
na_cells = int(data[FEATURE_COLS].isna().sum().sum())

n_dup = int(data.duplicated(subset=FEATURE_COLS + ["y_bin"]).sum())
data = data.drop_duplicates(subset=FEATURE_COLS + ["y_bin"], keep="first").reset_index(drop=True)

print(f"\nremoved duplicates: {n_dup:,} rows ({n_dup/n_rows_raw*100:.1f}% of raw)")
print(f"clean shape: {data.shape[0]:,} rows x {len(FEATURE_COLS)} features")
print(f"label counts: {data['y_bin'].value_counts().to_dict()}")
print(f"categories: {data['y_cat'].value_counts().to_dict()}")
print(f"elapsed: {time.time()-_t0:.1f}s")


### 2B · Triple-method duplicate verification

Three independent methods to defend the duplicate finding:
1. SHA-1 hash of feature bytes + label
2. `pandas.duplicated()` value-based
3. Tolerance-based pairwise check on a sample

In [ ]:
banner("SECTION 2B - TRIPLE-METHOD DUPLICATE VERIFICATION")
def row_hash(row):
    h = hashlib.sha1()
    h.update(row.tobytes())
    return h.hexdigest()

print("Method 1: hash-based (SHA-1)...")
_t0 = time.time()
hash_cols = FEATURE_COLS + ["y_bin"]
hash_matrix = data[hash_cols].copy()
for c in hash_cols:
    hash_matrix[c] = hash_matrix[c].astype('float64').fillna(-999999.0)
hash_values = hash_matrix.apply(lambda r: row_hash(r.values), axis=1)
n_unique_hash = hash_values.nunique()
n_dup_hash = len(data) - n_unique_hash
print(f"  unique hashes: {n_unique_hash:,} | dup by hash: {n_dup_hash:,} | time: {time.time()-_t0:.1f}s")

print("\nMethod 2: value-based (pandas.duplicated)...")
n_dup_value = int(data.duplicated(subset=FEATURE_COLS + ["y_bin"]).sum())
print(f"  dup by value: {n_dup_value:,}")

print("\nMethod 3: tolerance-based (sample 1000 rows)...")
_sample = data[FEATURE_COLS + ["y_bin"]].sample(n=min(1000, len(data)), random_state=SEED).reset_index(drop=True)
_arr = _sample.values.astype('float64')
_m = min(500, len(_arr))
_dup_pairs = sum(1 for i in range(_m) for j in range(i+1, _m) if (_arr[i] == _arr[j]).all())
print(f"  checked {_m*(_m-1)//2:,} pairs in sample of {_m} | exact matches: {_dup_pairs}")

methods_agree = (n_dup_hash == n_dup_value)
print(f"\n--- verification summary ---")
print(f"hash-based dup: {n_dup_hash:,}")
print(f"value-based dup: {n_dup_value:,}")
print(f"methods agree: {methods_agree}")
print(f"duplicate pct of raw: {n_dup_hash/n_rows_raw*100:.2f}%")

pd.Series({
    "raw_rows": n_rows_raw, "unique_rows_hash": n_unique_hash,
    "unique_rows_value": n_rows_raw - n_dup_value,
    "duplicate_rows_hash": n_dup_hash, "duplicate_rows_value": n_dup_value,
    "methods_agree": methods_agree,
    "duplicate_pct_of_raw": round(n_dup_hash / n_rows_raw * 100, 2),
}, name="value").to_csv(RESULTS_DIR / "duplicate_verification.csv")


### 2C · Per-class duplicate analysis

In [ ]:
banner("SECTION 2C - PER-CLASS DUPLICATE ANALYSIS")
raw2, _, cat2 = load_parts(csv_paths)
raw2 = raw2.drop(columns=[c for c in LEAK_COLS if c in raw2.columns])
raw2 = _slim(raw2)
raw2_cat = cat2 if cat2 is not None else pd.Series(['Unknown']*len(raw2))

per_class = []
for cls in sorted(raw2_cat.unique()):
    mask = (raw2_cat == cls).values
    cls_df = raw2.loc[mask]
    n_total = len(cls_df)
    n_dup = int(cls_df.duplicated().sum())
    per_class.append({"category": cls, "total_rows": n_total,
                      "duplicate_rows": n_dup,
                      "duplicate_pct": round(n_dup / n_total * 100, 2) if n_total > 0 else 0})
per_class_df = pd.DataFrame(per_class).sort_values("duplicate_pct", ascending=False)
per_class_df.to_csv(RESULTS_DIR / "per_class_duplicate_rates.csv", index=False)
print(per_class_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
colors = [CAT_COLORS.get(c, '#9E9E9E') for c in per_class_df['category']]
ax.barh(per_class_df['category'], per_class_df['duplicate_pct'], color=colors)
for i, v in enumerate(per_class_df['duplicate_pct']):
    ax.text(v + 1, i, f"{v:.1f}%", va='center', fontsize=9)
ax.set_xlabel("duplicate rate (%)"); ax.set_xlim(0, 100)
ax.set_title("Per-class duplicate rate in raw IoTID20")
finish_fig(fig, "fig_dup_per_class")
del raw2, raw2_cat; gc.collect()


### 2D · Class distribution & feature redundancy

In [ ]:
banner("SECTION 2D - DATA AUDIT")
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), constrained_layout=True)
bc = data["y_bin"].value_counts().reindex([0, 1])
bars = axes[0].bar(["Normal (0)", "Anomaly (1)"], bc.values,
                   color=[CAT_COLORS["Normal"], "#B2182B"], width=0.55)
for rect, v in zip(bars, bc.values):
    axes[0].annotate(f"{v:,}\n{v/len(data)*100:.1f}%", (rect.get_x()+rect.get_width()/2, v),
                     ha="center", va="bottom", fontsize=10)
axes[0].set_ylim(0, bc.max()*1.22); axes[0].set_ylabel("flows")
axes[0].set_title("Binary label distribution (after de-dup)")
cc = data["y_cat"].value_counts().sort_values()
axes[1].barh(cc.index, cc.values, color=[CAT_COLORS.get(c, "#9E9E9E") for c in cc.index])
for y, v in enumerate(cc.values):
    axes[1].annotate(f" {v:,} ({v/len(data)*100:.1f}%)", (v, y), va="center", fontsize=9)
axes[1].set_xlim(0, cc.max()*1.42)
axes[1].set_title("Traffic category distribution (after de-dup)")
finish_fig(fig, "fig01_class_distribution")

_corr_n = min(100_000, len(data))
corr = data[FEATURE_COLS].sample(n=_corr_n, random_state=SEED).corr().fillna(0.0)
abs_corr = corr.abs().to_numpy(copy=True)
np.fill_diagonal(abs_corr, 0.0)
redundant_pairs = sorted(
    ((FEATURE_COLS[i], FEATURE_COLS[j], float(corr.iloc[i, j]))
     for i, j in combinations(range(len(FEATURE_COLS)), 2) if abs_corr[i, j] > 0.95),
    key=lambda t: -abs(t[2]))
print(f"highly correlated feature pairs (|r|>0.95): {len(redundant_pairs)}")
for a, b, r in redundant_pairs[:10]:
    print(f"   {a}  ~  {b}   r={r:+.3f}")


## 3 · Leakage-safe split, imputation & scaling

In [ ]:
banner("SECTION 3 - SPLIT & SCALE")
X_all = data[FEATURE_COLS].astype(np.float32)
y_all = data["y_bin"]
cat_all = data["y_cat"].astype(str)
ts_all = data["_ts"]

_all_idx = np.arange(len(data))
idx_trainval, idx_test = train_test_split(_all_idx, test_size=TEST_FRAC, random_state=SEED, stratify=y_all)
idx_train, idx_val = train_test_split(idx_trainval, test_size=VAL_FRAC, random_state=SEED, stratify=y_all.iloc[idx_trainval])

def _take(idx):
    return X_all.iloc[idx].copy(), y_all.iloc[idx], cat_all.iloc[idx]

X_train, y_train, cat_train = _take(idx_train)
X_val, y_val, cat_val = _take(idx_val)
X_test, y_test, cat_test = _take(idx_test)
y_train_np = y_train.to_numpy(); y_val_np = y_val.to_numpy(); y_test_np = y_test.to_numpy()
cat_train_np = cat_train.to_numpy(); cat_test_np = cat_test.to_numpy()

# impute with TRAIN medians only — pandas 3.x safe (no inplace)
train_medians = X_train.median()
allnan_cols = train_medians.index[train_medians.isna()].tolist()
if allnan_cols:
    X_train = X_train.drop(columns=allnan_cols)
    X_val   = X_val.drop(columns=allnan_cols)
    X_test  = X_test.drop(columns=allnan_cols)
    print(f"dropped {len(allnan_cols)} all-missing feature(s): {allnan_cols}")
X_train = X_train.fillna(train_medians)
X_val   = X_val.fillna(train_medians)
X_test  = X_test.fillna(train_medians)

var0 = [c for c in X_train.columns if X_train[c].std(ddof=0) == 0]
if var0:
    X_train = X_train.drop(columns=var0)
    X_val   = X_val.drop(columns=var0)
    X_test  = X_test.drop(columns=var0)
    print(f"dropped {len(var0)} zero-variance features: {var0}")
FEATURE_COLS_KEEP = list(X_train.columns)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_val_s   = scaler.transform(X_val).astype(np.float32)
X_test_s  = scaler.transform(X_test).astype(np.float32)
if not np.isfinite(X_train_s).all():
    raise RuntimeError("non-finite values survived imputation + scaling")

print(f"splits -> train {X_train_s.shape[0]:,} | val {X_val_s.shape[0]:,} | test {X_test_s.shape[0]:,} | features {X_train_s.shape[1]}")
print(f"class dist: {np.bincount(y_train_np)} (train)")

# store raw (unscaled) versions for Priority 4 noise methodology
X_train_raw = X_train.to_numpy().astype(np.float32)
X_test_raw  = X_test.to_numpy().astype(np.float32)
feature_stds_raw = X_train.std(axis=0).to_numpy().astype(np.float32)
print(f"Stored raw (unscaled) feature matrices for noise methodology validation.")


## 4 · Trivial baselines

In [ ]:
banner("SECTION 4 - TRIVIAL BASELINES")
baseline_rows = []
for strategy in ["most_frequent", "stratified"]:
    dm = DummyClassifier(strategy=strategy, random_state=SEED).fit(X_train_s, y_train_np)
    p = dm.predict(X_test_s)
    baseline_rows.append({
        "Model": f"Dummy ({strategy})",
        "Accuracy": accuracy_score(y_test_np, p),
        "BalancedAcc": balanced_accuracy_score(y_test_np, p),
        "MCC": matthews_corrcoef(y_test_np, p),
        "F1_macro": f1_score(y_test_np, p, average="macro", zero_division=0),
    })
baselines_df = pd.DataFrame(baseline_rows)
baselines_df.to_csv(RESULTS_DIR / "dummy_baselines.csv", index=False)
print(baselines_df.round(4).to_string(index=False))


## 5 · Models

Three lightweight detectors. ⚠ In IoTID20 the **anomaly class is the majority**, so `scale_pos_weight = n_normal / n_anomaly < 1`.

In [ ]:
# =====================================================================
# SECTION 5A - LOGISTIC REGRESSION
# =====================================================================
banner("SECTION 5A - LOGISTIC REGRESSION")
def evaluate_binary(name, y_true, proba, train_time=None):
    proba = np.asarray(proba, dtype=np.float64)
    pred = (proba >= 0.5).astype(int)
    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, pred),
        "BalancedAcc": balanced_accuracy_score(y_true, pred),
        "Precision_anomaly": precision_score(y_true, pred, pos_label=1, zero_division=0),
        "Recall_anomaly": recall_score(y_true, pred, pos_label=1, zero_division=0),
        "F1_anomaly": f1_score(y_true, pred, pos_label=1, zero_division=0),
        "F1_macro": f1_score(y_true, pred, average="macro", zero_division=0),
        "MCC": matthews_corrcoef(y_true, pred),
        "ROC_AUC": roc_auc_score(y_true, proba),
        "PR_AUC": average_precision_score(y_true, proba),
        "Brier": brier_score_loss(y_true, proba),
        "train_sec": train_time,
    }
    RESULTS_ROWS.append(row); PROBS[name] = proba; PREDS[name] = pred
    print(f"[{name}] acc={row['Accuracy']:.4f} | balAcc={row['BalancedAcc']:.4f} | MCC={row['MCC']:.4f} | PR-AUC={row['PR_AUC']:.4f}")
    return row

lr = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", class_weight="balanced",
                        max_iter=400, random_state=SEED)
t0 = time.time()
lr.fit(X_train_s, y_train_np)
lr_train_time = time.time() - t0
evaluate_binary("Logistic Regression", y_test_np, lr.predict_proba(X_test_s)[:, 1], lr_train_time)
joblib.dump(lr, MODELS_DIR / "logistic_regression.joblib")


In [ ]:
# =====================================================================
# SECTION 5B - XGBOOST
# =====================================================================
banner("SECTION 5B - XGBOOST")
from xgboost import XGBClassifier
neg, pos = np.bincount(y_train_np)
spw = neg / pos
print(f"train class counts -> normal(0): {neg:,} | anomaly(1): {pos:,} | scale_pos_weight={spw:.3f}")
xgb_model = XGBClassifier(
    n_estimators=60 if FAST_MODE else 300,
    learning_rate=0.1, max_depth=6, subsample=0.9, colsample_bytree=0.8,
    objective="binary:logistic", eval_metric="logloss", tree_method="hist",
    scale_pos_weight=spw, early_stopping_rounds=25,
    random_state=SEED, n_jobs=-1,
)
t0 = time.time()
xgb_model.fit(X_train_s, y_train_np, eval_set=[(X_val_s, y_val_np)], verbose=False)
xgb_train_time = time.time() - t0
print(f"train time: {xgb_train_time:.1f}s | best iteration: {xgb_model.best_iteration}")
evaluate_binary("XGBoost", y_test_np, xgb_model.predict_proba(X_test_s)[:, 1], xgb_train_time)
joblib.dump(xgb_model, MODELS_DIR / "xgboost.joblib")


In [ ]:
# =====================================================================
# SECTION 5C - MLP (PyTorch, CPU)
# =====================================================================
banner("SECTION 5C - MLP (PyTorch)")
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch device: {device}")
X_tr_t = torch.from_numpy(X_train_s)
y_tr_t = torch.from_numpy(y_train_np.astype(np.float32)).unsqueeze(1)
X_va_t = torch.from_numpy(X_val_s)
y_va_t = torch.from_numpy(y_val_np.astype(np.float32)).unsqueeze(1)
X_te_t = torch.from_numpy(X_test_s)
class SmallNet(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 1),
        )
    def forward(self, x): return self.net(x)
mlp = SmallNet(X_tr_t.shape[1]).to(device)
pos_weight = torch.tensor([neg / pos], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-5)
loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=512, shuffle=True)
epochs_max = 5 if FAST_MODE else 15
patience = 2 if FAST_MODE else 3
history, best_val, best_state, wait = [], np.inf, None, 0
t0 = time.time()
for epoch in range(1, epochs_max + 1):
    mlp.train(); ep_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(mlp(xb), yb)
        loss.backward(); optimizer.step()
        ep_loss += loss.item() * xb.size(0)
    ep_loss /= len(loader.dataset)
    mlp.eval()
    with torch.no_grad():
        val_loss = criterion(mlp(X_va_t.to(device)), y_va_t.to(device)).item()
    history.append((epoch, ep_loss, val_loss))
    print(f"epoch {epoch:02d}/{epochs_max} | train {ep_loss:.4f} | val {val_loss:.4f}")
    if val_loss < best_val - 1e-4:
        best_val, wait = val_loss, 0
        best_state = {k: v.detach().cpu().clone() for k, v in mlp.state_dict().items()}
    else:
        wait += 1
        if wait >= patience:
            print(f"early stopping at epoch {epoch}"); break
mlp_train_time = time.time() - t0
if best_state is not None:
    mlp.load_state_dict(best_state)
mlp.eval()
torch.save(mlp.state_dict(), MODELS_DIR / "mlp_state_dict.pt")
print(f"train time: {mlp_train_time:.1f}s | parameters: {sum(p.numel() for p in mlp.parameters()):,}")
with torch.no_grad():
    probs_nn = torch.sigmoid(mlp(X_te_t.to(device))).cpu().numpy().ravel()
evaluate_binary("Neural Network", y_test_np, probs_nn, mlp_train_time)
pd.DataFrame(history, columns=["epoch","train_loss","val_loss"]).to_csv(RESULTS_DIR / "mlp_training_history.csv", index=False)


In [ ]:
# SECTION 5D - METRICS SUMMARY
model_names = ["Logistic Regression", "XGBoost", "Neural Network"]
metrics_df = pd.DataFrame(RESULTS_ROWS)
metrics_df.to_csv(RESULTS_DIR / "binary_metrics_test.csv", index=False)
print(metrics_df.set_index("Model").round(4).to_string())


## 6 · Core evaluation

In [ ]:
banner("SECTION 6A - CONFUSION MATRICES")
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), constrained_layout=True)
for ax, name in zip(axes, model_names):
    cm = confusion_matrix(y_test_np, PREDS[name])
    pct = cm / cm.sum(axis=1, keepdims=True) * 100
    annot = np.array([[f"{v:,.0f}\n({p:.1f}%)" for v, p in zip(rv, rp)] for rv, rp in zip(cm, pct)])
    sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Normal","Anomaly"], yticklabels=["Normal","Anomaly"],
                annot_kws={"fontsize": 9})
    ax.set_title(f"{name}\nMCC={matthews_corrcoef(y_test_np, PREDS[name]):.3f}")
    ax.set_xlabel("predicted"); ax.set_ylabel("actual")
finish_fig(fig, "fig05_confusion_matrices")


In [ ]:
banner("SECTION 6B - ROC & PR CURVES")
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.6), constrained_layout=True)
for name in model_names:
    c = MODEL_COLORS[name]
    fpr, tpr, _ = roc_curve(y_test_np, PROBS[name])
    axes[0].plot(fpr, tpr, color=c, lw=2, label=f"{name} (AUC={roc_auc_score(y_test_np, PROBS[name]):.4f})")
    prec, rec, _ = precision_recall_curve(y_test_np, PROBS[name])
    axes[1].plot(rec, prec, color=c, lw=2, label=f"{name} (AP={average_precision_score(y_test_np, PROBS[name]):.4f})")
axes[0].plot([0,1],[0,1],"k--",lw=1,alpha=0.6,label="chance")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("ROC - anomaly class")
axes[0].legend(fontsize=8); axes[0].set_xlim(-0.02,1.02); axes[0].set_ylim(0,1.05)
prev = y_test_np.mean()
axes[1].axhline(prev, color="k", ls="--", lw=1, alpha=0.6, label=f"prevalence ({prev:.2f})")
axes[1].set_xlabel("recall (anomaly)"); axes[1].set_ylabel("precision (anomaly)")
axes[1].set_title("PR - anomaly class"); axes[1].legend(fontsize=8)
axes[1].set_xlim(-0.02,1.02); axes[1].set_ylim(0,1.05)
y0 = (y_test_np == 0).astype(int)
for name in model_names:
    prec0, rec0, _ = precision_recall_curve(y0, 1.0 - PROBS[name])
    axes[2].plot(rec0, prec0, color=MODEL_COLORS[name], lw=2,
                 label=f"{name} (AP={average_precision_score(y0, 1.0 - PROBS[name]):.3f})")
axes[2].axhline(y0.mean(), color="k", ls="--", lw=1, alpha=0.6)
axes[2].set_xlabel("recall (Normal)"); axes[2].set_ylabel("precision (Normal)")
axes[2].set_title("PR - minority Normal view"); axes[2].legend(fontsize=8)
axes[2].set_xlim(-0.02,1.02); axes[2].set_ylim(0,1.05)
finish_fig(fig, "fig06_roc_pr_curves")


In [ ]:
banner("SECTION 6C - CALIBRATION")
fig, ax = plt.subplots(figsize=(7.5, 5.5), constrained_layout=True)
for name in model_names:
    frac_pos, mean_pred = calibration_curve(y_test_np, PROBS[name], n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, "o-", color=MODEL_COLORS[name], lw=2, label=name)
ax.plot([0,1],[0,1],"k--",lw=1,alpha=0.6,label="perfect")
ax.set_xlabel("mean predicted prob"); ax.set_ylabel("empirical anomaly rate")
ax.set_title("Calibration"); ax.legend(fontsize=9)
finish_fig(fig, "fig07_calibration")


## 7 · Per-attack-family analysis

Overall MCC tells us *that* a model works; per-family F1 tells us *where*.

In [ ]:
banner("SECTION 7 - PER-ATTACK FAMILY ANALYSIS")
best_model_name = "XGBoost"
test_pred = PREDS[best_model_name]
per_family_rows = []
for fam in sorted(np.unique(cat_test_np)):
    mask = (cat_test_np == fam)
    if mask.sum() == 0: continue
    y_fam = y_test_np[mask]
    p_fam = test_pred[mask]
    is_normal = (fam == 'Normal')
    if is_normal:
        correct = (p_fam == 0)
        precision = correct.sum() / max(1, (p_fam == 0).sum())
        recall = correct.sum() / max(1, len(y_fam))
    else:
        correct = (p_fam == 1)
        precision = (correct & (y_fam == 1)).sum() / max(1, (p_fam == 1).sum())
        recall = correct.sum() / max(1, len(y_fam))
    f1_fam = 2 * precision * recall / max(1e-9, precision + recall)
    per_family_rows.append({"family": fam, "n_flows": int(mask.sum()),
                            "precision": round(precision,4), "recall": round(recall,4),
                            "F1": round(f1_fam,4)})
per_family_df = pd.DataFrame(per_family_rows).sort_values("F1")
per_family_df.to_csv(RESULTS_DIR / "per_attack_family_metrics.csv", index=False)
print(per_family_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
colors = [CAT_COLORS.get(f, '#9E9E9E') for f in per_family_df['family']]
ax.barh(per_family_df['family'], per_family_df['F1'], color=colors)
for i, row in per_family_df.reset_index(drop=True).iterrows():
    ax.text(row['F1'] + 0.01, i, f"F1={row['F1']:.3f}", va='center', fontsize=9)
ax.set_xlabel("F1 (XGBoost)"); ax.set_xlim(0, 1.1)
ax.set_title("Per-attack-family detection difficulty")
finish_fig(fig, "fig08_per_family")


## 8 · Efficiency benchmark

In [ ]:
banner("SECTION 8 - EFFICIENCY BENCHMARK")
import pickle
def measure_latency(predict_fn, X, n=LATENCY_N):
    for _ in range(min(20, n // 4)): predict_fn(X[:1])
    times = []
    for _ in range(n):
        i = np.random.randint(0, len(X))
        t0 = time.perf_counter()
        predict_fn(X[i:i+1])
        times.append(time.perf_counter() - t0)
    return float(np.mean(times) * 1e6)
def measure_throughput(predict_fn, X, batch=BATCH_N):
    t0 = time.perf_counter()
    predict_fn(X[:batch])
    return batch / (time.perf_counter() - t0)
lr_fn = lambda x: lr.predict_proba(x)[:, 1]
xgb_fn = lambda x: xgb_model.predict_proba(x)[:, 1]
def mlp_fn(x):
    with torch.no_grad():
        return torch.sigmoid(mlp(torch.from_numpy(x).to(device))).cpu().numpy().ravel()
eff_rows = [
    {"Model": "Logistic Regression", "latency_us": measure_latency(lr_fn, X_test_s),
     "throughput_fps": measure_throughput(lr_fn, X_test_s),
     "model_size_KB": len(pickle.dumps(lr))/1024, "train_sec": lr_train_time},
    {"Model": "XGBoost", "latency_us": measure_latency(xgb_fn, X_test_s),
     "throughput_fps": measure_throughput(xgb_fn, X_test_s),
     "model_size_KB": len(pickle.dumps(xgb_model))/1024, "train_sec": xgb_train_time},
    {"Model": "Neural Network", "latency_us": measure_latency(mlp_fn, X_test_s),
     "throughput_fps": measure_throughput(mlp_fn, X_test_s),
     "model_size_KB": sum(p.numel()*p.element_size() for p in mlp.parameters())/1024, "train_sec": mlp_train_time},
]
eff_df = pd.DataFrame(eff_rows)
eff_df.to_csv(RESULTS_DIR / "efficiency_benchmark.csv", index=False)
print(eff_df.round(2).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
for _, r in eff_df.iterrows():
    c = MODEL_COLORS[r['Model']]
    mcc_v = metrics_df.set_index('Model').loc[r['Model'], 'MCC']
    axes[0].scatter(r['latency_us'], mcc_v, s=r['model_size_KB']*2, color=c, alpha=0.7, edgecolors='black')
    axes[0].annotate(r['Model'], (r['latency_us'], mcc_v), xytext=(5,5), textcoords='offset points', fontsize=9)
axes[0].set_xlabel("latency (us)"); axes[0].set_ylabel("MCC"); axes[0].set_xscale('log')
axes[0].set_title("Accuracy vs efficiency (bubble=size KB)")
colors_list = [MODEL_COLORS[m] for m in eff_df['Model']]
axes[1].bar(eff_df['Model'], eff_df['throughput_fps'], color=colors_list)
axes[1].set_ylabel("throughput (flows/sec)"); axes[1].set_title("Batch throughput")
axes[1].tick_params(axis='x', rotation=15)
finish_fig(fig, "fig09_efficiency")


## 9 · SHAP explainability

In [ ]:
banner("SECTION 9 - SHAP")
import shap
X_test_df = pd.DataFrame(X_test_s, columns=FEATURE_COLS_KEEP)
explainer_xgb = shap.TreeExplainer(xgb_model)
shap_n = min(SHAP_SAMPLE, len(X_test_s))
shap_vals_xgb = explainer_xgb.shap_values(X_test_df.iloc[:shap_n])
if isinstance(shap_vals_xgb, list):
    shap_arr = shap_vals_xgb[1]
else:
    shap_arr = shap_vals_xgb if shap_vals_xgb.ndim == 2 else shap_vals_xgb[..., 1]
importance = np.abs(shap_arr).mean(axis=0)
imp_df = pd.Series(importance, index=FEATURE_COLS_KEEP, name="shap_abs_mean").sort_values(ascending=False)
imp_df.to_csv(RESULTS_DIR / "shap_feature_importance.csv")

fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
top_n = 15
ax.barh(imp_df.head(top_n).index[::-1], imp_df.head(top_n).values[::-1], color=MODEL_COLORS["XGBoost"])
ax.set_xlabel("mean |SHAP value|"); ax.set_title(f"Top {top_n} features (XGBoost)")
finish_fig(fig, "fig10_shap_importance")
print("Top 10 features:")
print(imp_df.head(10).round(4).to_string())


## 10 · Feature budget

In [ ]:
banner("SECTION 10 - FEATURE BUDGET")
ranked = imp_df.index.tolist()
fb_rows = []
for k in FEATURE_KS:
    k_eff = min(k, len(ranked))
    if k_eff == 0: continue
    cols_k = ranked[:k_eff]
    col_idx = [FEATURE_COLS_KEEP.index(c) for c in cols_k]
    m = XGBClassifier(n_estimators=60 if FAST_MODE else 200, learning_rate=0.1, max_depth=6,
                      subsample=0.9, colsample_bytree=0.8, objective="binary:logistic",
                      eval_metric="logloss", tree_method="hist", scale_pos_weight=spw,
                      early_stopping_rounds=20, random_state=SEED, n_jobs=-1)
    m.fit(X_train_s[:, col_idx], y_train_np, eval_set=[(X_val_s[:, col_idx], y_val_np)], verbose=False)
    p = m.predict_proba(X_test_s[:, col_idx])[:, 1]
    fb_rows.append({"k": k_eff, "MCC": matthews_corrcoef(y_test_np, (p>=0.5).astype(int)),
                    "PR_AUC": average_precision_score(y_test_np, p)})
fb_df = pd.DataFrame(fb_rows)
fb_df.to_csv(RESULTS_DIR / "feature_budget.csv", index=False)
print(fb_df.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
ax.plot(fb_df['k'], fb_df['MCC'], 'o-', color=MODEL_COLORS['XGBoost'], label='MCC')
ax.plot(fb_df['k'], fb_df['PR_AUC'], 's--', color='#555', label='PR-AUC')
ax.set_xlabel("# features"); ax.set_ylabel("score")
ax.set_title("Feature budget"); ax.legend(); ax.grid(True, alpha=0.3)
finish_fig(fig, "fig11_feature_budget")


## 11 · Cost-sensitive threshold

In [ ]:
banner("SECTION 11 - COST-SENSITIVE THRESHOLD")
def expected_cost(y_true, proba, t, c_fn, c_fp):
    pred = (proba >= t).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return c_fn * fn + c_fp * fp
thresholds = np.linspace(0.01, 0.99, 99)
cost_rows = []
for name in model_names:
    costs = [expected_cost(y_test_np, PROBS[name], t, COST_FN, COST_FP) for t in thresholds]
    best_idx = int(np.argmin(costs))
    default_cost = expected_cost(y_test_np, PROBS[name], 0.5, COST_FN, COST_FP)
    cost_rows.append({"Model": name, "default_cost": default_cost,
                      "optimal_threshold": round(float(thresholds[best_idx]), 3),
                      "optimal_cost": float(costs[best_idx]),
                      "cost_reduction_pct": round((1 - costs[best_idx]/default_cost)*100, 2) if default_cost>0 else 0})
cost_df = pd.DataFrame(cost_rows)
cost_df.to_csv(RESULTS_DIR / "cost_sensitive_thresholds.csv", index=False)
print(cost_df.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
for name in model_names:
    costs = [expected_cost(y_test_np, PROBS[name], t, COST_FN, COST_FP) for t in thresholds]
    ax.plot(thresholds, costs, color=MODEL_COLORS[name], lw=2, label=name)
ax.axvline(0.5, color='k', ls=':', alpha=0.5)
ax.set_xlabel("threshold"); ax.set_ylabel(f"expected cost")
ax.set_title("Cost-sensitive threshold tuning"); ax.legend(fontsize=9)
finish_fig(fig, "fig12_cost_threshold")


## 12 · Robustness stress tests

**Priority 4 fix**: the original notebook added noise to *standardized* features with sigma=0.1, which is huge (10% of std on every feature simultaneously). Here we use two proper methodologies:

- **Method A (raw-feature noise)**: add Gaussian noise to raw features *before* scaling, with sigma relative to each feature's std. This simulates real measurement noise.
- **Method B (scaled-feature noise)**: the original method, kept for comparison, with much smaller sigma levels.

If the two methods disagree dramatically, the original finding was a methodology artifact.

In [ ]:
# =====================================================================
# SECTION 12A - NOISE ROBUSTNESS (FIXED METHODOLOGY)
# =====================================================================
banner("SECTION 12A - NOISE ROBUSTNESS (FIXED METHODOLOGY)")

def predict_proba_safe(model, X, name):
    if name == 'Neural Network':
        with torch.no_grad():
            return torch.sigmoid(model(torch.from_numpy(X).to(device))).cpu().numpy().ravel()
    return model.predict_proba(X)[:, 1]

# ---- Method A: RAW-FEATURE noise (sigma relative to each feature's std) ----
print("--- Method A: raw-feature noise (sigma_relative = fraction of feature std) ---")
noise_a_rows = []
for sigma_rel in NOISE_SIGMAS:
    rng = np.random.RandomState(SEED)
    # noise in raw space, proportional to per-feature std
    noise = rng.normal(0, 1, X_test_raw.shape).astype(np.float32) * (sigma_rel * feature_stds_raw)
    X_test_raw_noisy = X_test_raw + noise
    # scale the noisy raw features with the SAME scaler fit on clean train
    X_test_noisy_s = scaler.transform(X_test_raw_noisy).astype(np.float32)
    for name, mdl in [('Logistic Regression', lr), ('XGBoost', xgb_model), ('Neural Network', mlp)]:
        p = predict_proba_safe(mdl, X_test_noisy_s, name)
        noise_a_rows.append({
            "method": "A_raw_relative", "sigma": sigma_rel, "Model": name,
            "MCC": matthews_corrcoef(y_test_np, (p >= 0.5).astype(int)),
            "Accuracy": accuracy_score(y_test_np, (p >= 0.5).astype(int)),
        })
noise_a_df = pd.DataFrame(noise_a_rows)
noise_a_df.to_csv(RESULTS_DIR / "noise_robustness_method_A.csv", index=False)
print(noise_a_df.round(4).to_string(index=False))

# ---- Method B: SCALED-FEATURE noise (original method, smaller sigmas) ----
print("\n--- Method B: scaled-feature noise (original method) ---")
noise_b_rows = []
for sigma in NOISE_SIGMAS:
    rng = np.random.RandomState(SEED)
    X_test_noisy = X_test_s + rng.normal(0, sigma, X_test_s.shape).astype(np.float32)
    for name, mdl in [('Logistic Regression', lr), ('XGBoost', xgb_model), ('Neural Network', mlp)]:
        p = predict_proba_safe(mdl, X_test_noisy, name)
        noise_b_rows.append({
            "method": "B_scaled_absolute", "sigma": sigma, "Model": name,
            "MCC": matthews_corrcoef(y_test_np, (p >= 0.5).astype(int)),
            "Accuracy": accuracy_score(y_test_np, (p >= 0.5).astype(int)),
        })
noise_b_df = pd.DataFrame(noise_b_rows)
noise_b_df.to_csv(RESULTS_DIR / "noise_robustness_method_B.csv", index=False)
print(noise_b_df.round(4).to_string(index=False))

# ---- Comparison plot ----
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
for name in model_names:
    sub_a = noise_a_df[noise_a_df['Model'] == name]
    sub_b = noise_b_df[noise_b_df['Model'] == name]
    axes[0].plot(sub_a['sigma'], sub_a['MCC'], 'o-', color=MODEL_COLORS[name], lw=2, label=name)
    axes[1].plot(sub_b['sigma'], sub_b['MCC'], 's--', color=MODEL_COLORS[name], lw=2, label=name)
axes[0].set_xlabel("sigma (relative to feature std)"); axes[0].set_ylabel("MCC")
axes[0].set_title("Method A: raw-feature noise (proper)"); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel("sigma (absolute, on scaled features)"); axes[1].set_ylabel("MCC")
axes[1].set_title("Method B: scaled-feature noise (original)"); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
finish_fig(fig, "fig13_noise_robustness_comparison")

print("\n--- Diagnosis ---")
# Compare XGBoost at sigma=0.1 across methods
xb_a = noise_a_df[(noise_a_df['Model']=='XGBoost') & (noise_a_df['sigma']==0.1)]['MCC'].values
xb_b = noise_b_df[(noise_b_df['Model']=='XGBoost') & (noise_b_df['sigma']==0.1)]['MCC'].values
if len(xb_a) and len(xb_b):
    print(f"XGBoost at sigma=0.1: Method A (raw) MCC={xb_a[0]:.4f} vs Method B (scaled) MCC={xb_b[0]:.4f}")
    print("If Method A >> Method B, the original drop was a methodology artifact (noise on standardized features is unrealistically large).")


In [ ]:
# =====================================================================
# SECTION 12B - FEATURE DROPOUT
# =====================================================================
banner("SECTION 12B - FEATURE DROPOUT")
drop_rows = []
for pct in DROPOUT_PCTS:
    rng = np.random.RandomState(SEED)
    X_drop = X_test_s.copy()
    if pct > 0:
        n_drop = int(X_drop.shape[1] * pct / 100)
        cols = rng.choice(X_drop.shape[1], n_drop, replace=False)
        X_drop[:, cols] = 0.0
    for name, mdl in [('Logistic Regression', lr), ('XGBoost', xgb_model), ('Neural Network', mlp)]:
        p = predict_proba_safe(mdl, X_drop, name)
        drop_rows.append({"dropout_pct": pct, "Model": name,
                          "MCC": matthews_corrcoef(y_test_np, (p>=0.5).astype(int))})
drop_df = pd.DataFrame(drop_rows)
drop_df.to_csv(RESULTS_DIR / "feature_dropout_robustness.csv", index=False)
print(drop_df.round(4).to_string(index=False))


In [ ]:
# =====================================================================
# SECTION 12C - LABEL NOISE
# =====================================================================
banner("SECTION 12C - LABEL NOISE")
ln_rows = []
for noise_rate in LABEL_NOISE_RATES:
    rng = np.random.RandomState(SEED)
    y_noisy = y_train_np.copy()
    n_flip = int(len(y_noisy) * noise_rate)
    if n_flip > 0:
        flip_idx = rng.choice(len(y_noisy), n_flip, replace=False)
        y_noisy[flip_idx] = 1 - y_noisy[flip_idx]
    m = XGBClassifier(n_estimators=60 if FAST_MODE else 200, learning_rate=0.1, max_depth=6,
                      subsample=0.9, colsample_bytree=0.8, objective="binary:logistic",
                      eval_metric="logloss", tree_method="hist", scale_pos_weight=spw,
                      early_stopping_rounds=20, random_state=SEED, n_jobs=-1)
    m.fit(X_train_s, y_noisy, eval_set=[(X_val_s, y_val_np)], verbose=False)
    p = m.predict_proba(X_test_s)[:, 1]
    ln_rows.append({"label_noise_rate": noise_rate,
                    "MCC": matthews_corrcoef(y_test_np, (p>=0.5).astype(int)),
                    "PR_AUC": average_precision_score(y_test_np, p)})
ln_df = pd.DataFrame(ln_rows)
ln_df.to_csv(RESULTS_DIR / "label_noise_robustness.csv", index=False)
print(ln_df.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
ax.plot(ln_df['label_noise_rate'], ln_df['MCC'], 'o-', color=MODEL_COLORS['XGBoost'], label='MCC')
ax.plot(ln_df['label_noise_rate'], ln_df['PR_AUC'], 's--', color='#555', label='PR-AUC')
ax.set_xlabel("training label noise rate"); ax.set_ylabel("score")
ax.set_title("Label noise robustness (XGBoost)"); ax.legend(); ax.grid(True, alpha=0.3)
finish_fig(fig, "fig15_label_noise")


In [ ]:
# =====================================================================
# SECTION 12D - CONCEPT DRIFT / TEMPORAL SPLIT
# =====================================================================
banner("SECTION 12D - CONCEPT DRIFT")
ts_valid = data['_ts'].notna()
temporal_mcc = None
if ts_valid.sum() > 0 and ts_valid.sum() == len(data):
    ts_sorted_idx = data['_ts'].sort_values().index
    n_tr_t = int(0.7 * len(ts_sorted_idx))
    tr_t_idx = ts_sorted_idx[:n_tr_t]
    te_t_idx = ts_sorted_idx[n_tr_t:]
    X_tr_t = X_all.iloc[tr_t_idx].copy()
    y_tr_t = y_all.iloc[tr_t_idx].to_numpy()
    X_te_t = X_all.iloc[te_t_idx].copy()
    y_te_t = y_all.iloc[te_t_idx].to_numpy()
    t_med = X_tr_t.median()
    allnan_t = t_med.index[t_med.isna()].tolist()
    if allnan_t:
        X_tr_t = X_tr_t.drop(columns=allnan_t)
        X_te_t = X_te_t.drop(columns=allnan_t)
    X_tr_t = X_tr_t.fillna(t_med)
    X_te_t = X_te_t.fillna(t_med)
    common_cols = [c for c in X_tr_t.columns if c in FEATURE_COLS_KEEP]
    X_tr_t = X_tr_t[common_cols]
    X_te_t = X_te_t[common_cols]
    sc_t = StandardScaler().fit(X_tr_t)
    X_tr_t_s = sc_t.transform(X_tr_t).astype(np.float32)
    X_te_t_s = sc_t.transform(X_te_t).astype(np.float32)
    neg_t, pos_t = np.bincount(y_tr_t)
    m_t = XGBClassifier(n_estimators=60 if FAST_MODE else 200, learning_rate=0.1, max_depth=6,
                        subsample=0.9, colsample_bytree=0.8, objective="binary:logistic",
                        eval_metric="logloss", tree_method="hist", scale_pos_weight=neg_t/pos_t,
                        early_stopping_rounds=20, random_state=SEED, n_jobs=-1)
    n_val_t = int(0.15 * len(X_tr_t_s))
    m_t.fit(X_tr_t_s[:-n_val_t], y_tr_t[:-n_val_t],
            eval_set=[(X_tr_t_s[-n_val_t:], y_tr_t[-n_val_t:])], verbose=False)
    p_t = m_t.predict_proba(X_te_t_s)[:, 1]
    temporal_mcc = matthews_corrcoef(y_te_t, (p_t >= 0.5).astype(int))
    random_mcc = metrics_df.set_index('Model').loc['XGBoost', 'MCC']
    print(f"Random split MCC:   {random_mcc:.4f}")
    print(f"Temporal split MCC: {temporal_mcc:.4f}")
    print(f"Optimism: {random_mcc - temporal_mcc:.4f}")
    pd.Series({'random_mcc': random_mcc, 'temporal_mcc': temporal_mcc,
               'optimism': random_mcc - temporal_mcc}).to_csv(RESULTS_DIR / "temporal_split.csv")
    fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
    ax.bar(['Random', 'Temporal'], [random_mcc, temporal_mcc],
           color=[MODEL_COLORS['XGBoost'], '#777777'])
    for i, v in enumerate([random_mcc, temporal_mcc]):
        ax.text(i, v + 0.01, f"{v:.4f}", ha='center')
    ax.set_ylabel("MCC"); ax.set_title("Random vs temporal split")
    ax.set_ylim(0, 1.05)
    finish_fig(fig, "fig16_temporal_split")
else:
    print("Timestamps not available - skipping.")


## 13 · Adversarial attacks (FGSM, PGD, feature perturbation)

White-box attacks on MLP (FGSM, PGD) and black-box feature perturbation on XGBoost.

In [ ]:
banner("SECTION 13 - ADVERSARIAL ATTACKS")
def fgsm_attack(model, X_t, y_t, eps, crit):
    X_adv = X_t.clone().detach().requires_grad_(True)
    loss = crit(model(X_adv), y_t)
    model.zero_grad(); loss.backward()
    return (X_adv + eps * X_adv.grad.sign()).detach()

def pgd_attack(model, X_t, y_t, eps, alpha, steps, crit):
    X_adv = X_t.clone().detach() + torch.empty_like(X_t).uniform_(-eps, eps)
    for _ in range(steps):
        X_adv.requires_grad_(True)
        loss = crit(model(X_adv), y_t)
        model.zero_grad(); loss.backward()
        with torch.no_grad():
            X_adv = X_adv + alpha * X_adv.grad.sign()
            X_adv = torch.max(torch.min(X_adv, X_t + eps), X_t - eps)
    return X_adv.detach()

adv_n = min(2000 if not FAST_MODE else 500, len(X_test_s))
rng_a = np.random.RandomState(SEED)
adv_idx = rng_a.choice(len(X_test_s), adv_n, replace=False)
X_adv_np = X_test_s[adv_idx]
y_adv_np = y_test_np[adv_idx]
anom_mask = (y_adv_np == 1)
X_adv_anom = X_adv_np[anom_mask]
y_adv_anom = y_adv_np[anom_mask]
print(f"Adv sample: {adv_n} total, {anom_mask.sum()} anomalies")
X_adv_t = torch.from_numpy(X_adv_anom).float().to(device)
y_adv_t = torch.from_numpy(y_adv_anom.astype(np.float32)).unsqueeze(1).to(device)
with torch.no_grad():
    p_clean = torch.sigmoid(mlp(X_adv_t)).cpu().numpy().ravel()
clean_tpr = float((p_clean >= 0.5).mean())
print(f"Baseline MLP TPR: {clean_tpr:.4f}")

adv_rows = []
for eps in FGSM_EPSILONS:
    if eps == 0:
        adv_rows.append({"attack":"FGSM","epsilon":eps,"TPR_after_attack":clean_tpr,"TPR_drop":0.0})
        adv_rows.append({"attack":"PGD","epsilon":eps,"TPR_after_attack":clean_tpr,"TPR_drop":0.0})
        continue
    X_fgsm = fgsm_attack(mlp, X_adv_t, y_adv_t, eps, criterion)
    with torch.no_grad():
        p_fgsm = torch.sigmoid(mlp(X_fgsm)).cpu().numpy().ravel()
    tpr_f = float((p_fgsm >= 0.5).mean())
    adv_rows.append({"attack":"FGSM","epsilon":eps,"TPR_after_attack":tpr_f,"TPR_drop":clean_tpr-tpr_f})
    X_pgd = pgd_attack(mlp, X_adv_t, y_adv_t, eps, eps/PGD_STEPS, PGD_STEPS, criterion)
    with torch.no_grad():
        p_pgd = torch.sigmoid(mlp(X_pgd)).cpu().numpy().ravel()
    tpr_p = float((p_pgd >= 0.5).mean())
    adv_rows.append({"attack":"PGD","epsilon":eps,"TPR_after_attack":tpr_p,"TPR_drop":clean_tpr-tpr_p})
adv_df = pd.DataFrame(adv_rows)
adv_df.to_csv(RESULTS_DIR / "adversarial_attack_results.csv", index=False)
print(adv_df.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
for atk in ['FGSM','PGD']:
    sub = adv_df[adv_df['attack']==atk]
    ax.plot(sub['epsilon'], sub['TPR_after_attack'], 'o-',
            color=MODEL_COLORS['Neural Network'] if atk=='FGSM' else '#D55E00', lw=2, label=f'MLP under {atk}')
ax.axhline(clean_tpr, color='k', ls=':', alpha=0.5)
ax.set_xlabel("epsilon"); ax.set_ylabel("TPR after attack")
ax.set_title("Adversarial robustness: FGSM vs PGD"); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
finish_fig(fig, "fig17_adversarial_mlp")


In [ ]:
banner("SECTION 13B - FEATURE PERTURBATION ON XGBOOST")
if len(imp_df) > 0:
    top_feat = imp_df.index[0]
    top_idx = FEATURE_COLS_KEEP.index(top_feat)
    print(f"Perturbing top feature '{top_feat}'")
    p_base = xgb_model.predict_proba(X_adv_anom)[:, 1]
    tpr_base = float((p_base >= 0.5).mean())
    xb_rows = []
    for eps in FGSM_EPSILONS:
        if eps == 0:
            xb_rows.append({"epsilon":eps,"TPR_after_attack":tpr_base,"TPR_drop":0.0})
            continue
        Xp = X_adv_anom.copy()
        Xp[:, top_idx] += eps
        p_pos = xgb_model.predict_proba(Xp)[:, 1]
        Xp2 = X_adv_anom.copy()
        Xp2[:, top_idx] -= eps
        p_neg = xgb_model.predict_proba(Xp2)[:, 1]
        tpr_adv = min(float((p_pos>=0.5).mean()), float((p_neg>=0.5).mean()))
        xb_rows.append({"epsilon":eps,"TPR_after_attack":tpr_adv,"TPR_drop":tpr_base-tpr_adv})
    xb_df = pd.DataFrame(xb_rows)
    xb_df.to_csv(RESULTS_DIR / "xgb_adversarial.csv", index=False)
    print(xb_df.round(4).to_string(index=False))
    fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    ax.plot(xb_df['epsilon'], xb_df['TPR_after_attack'], 'o-', color=MODEL_COLORS['XGBoost'], label='XGBoost')
    mlp_fgsm = adv_df[adv_df['attack']=='FGSM']
    ax.plot(mlp_fgsm['epsilon'], mlp_fgsm['TPR_after_attack'], 's--', color=MODEL_COLORS['Neural Network'], label='MLP FGSM')
    ax.axhline(tpr_base, color='k', ls=':', alpha=0.5)
    ax.set_xlabel("epsilon"); ax.set_ylabel("TPR")
    ax.set_title("Adversarial: tree vs MLP"); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    finish_fig(fig, "fig18_adversarial_xgb_vs_mlp")


## 14 · Cross-dataset validation (Priority 2)

Auto-detects TON_IoT, BoT_IoT, or Edge-IIoTset under `data/`. For each found dataset:
1. **Intra-dataset**: train+test on the new dataset — does the model ranking replicate?
2. **Cross-dataset (transfer)**: apply IoTID20-trained XGBoost directly — how badly does it degrade?

Feature harmonization: only features with the same name across datasets are used, to avoid semantic mismatch.

In [ ]:
banner("SECTION 14 - CROSS-DATASET VALIDATION")

def find_alt_datasets(base=DATA_DIR):
    found = {}
    for p in Path(base).rglob('*.csv'):
        nl = p.name.lower()
        if 'ton_iot' in nl or 'ton-iot' in nl or 'toniot' in nl:
            found['TON_IoT'] = p
        elif 'bot_iot' in nl or 'bot-iot' in nl or 'botiot' in nl:
            found['BoT_IoT'] = p
        elif 'edge' in nl and ('iiot' in nl or 'iot' in nl):
            found['Edge-IIoTset'] = p
    return found

alt_datasets = find_alt_datasets()
cross_results = []

if not alt_datasets or not RUN_CROSS_DATASET:
    print("No alternative datasets found under ./data/.")
    print("To enable cross-dataset validation, place any of:")
    print("  - TON_IoT CSV (from kaggle gwdaraktle/toniot-datasets)")
    print("  - BoT_IoT CSV (from kaggle aldostronsky/botiot)")
    print("  - Edge-IIoTset CSV (from kaggle mohamedaminebouzid/edge-iiotset-...)")
    print("inside ./data/ and re-run.")
else:
    print(f"Found alternative datasets: {list(alt_datasets.keys())}")
    for ds_name, ds_path in alt_datasets.items():
        print(f"\n=== Processing {ds_name} ({ds_path.name}) ===")
        try:
            df_alt = pd.read_csv(ds_path, low_memory=False, nrows=50_000 if FAST_MODE else None)
            print(f"  loaded: {len(df_alt):,} rows x {len(df_alt.columns)} cols")
            # find label column
            label_col = None
            for c in df_alt.columns:
                if c.lower() in ('label','label_binary','attack_label','class','category'):
                    label_col = c; break
            if label_col is None:
                print(f"  WARNING: no label column found in {ds_name}, skipping")
                continue
            print(f"  label column: '{label_col}'")
            # build binary label (1=anomaly/attack)
            y_alt_raw = df_alt[label_col]
            if y_alt_raw.dtype == object:
                y_alt = y_alt_raw.astype(str).str.lower().isin(['anomaly','1','attack','malicious']).astype(int).to_numpy()
            else:
                y_alt = (y_alt_raw != 0).astype(int).to_numpy()
            df_alt = df_alt.drop(columns=[label_col])
            # drop non-numeric and leaky columns
            for c in df_alt.columns:
                if df_alt[c].dtype == object:
                    df_alt = df_alt.drop(columns=c)
            df_alt = df_alt.replace([np.inf, -np.inf], np.nan)
            # find common features with IoTID20
            alt_features = set(df_alt.columns)
            id20_features = set(FEATURE_COLS_KEEP)
            common = sorted(alt_features & id20_features)
            print(f"  features: alt={len(alt_features)}, iotid20={len(id20_features)}, common={len(common)}")
            if len(common) < 5:
                print(f"  too few common features, using all alt features")
                common = sorted(alt_features)
            X_alt = df_alt[common].fillna(df_alt[common].median())
            X_alt = X_alt.astype(np.float32)
            # INTRA-DATASET: train+test on alt
            Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(X_alt, y_alt, test_size=0.25, random_state=SEED, stratify=y_alt if len(np.unique(y_alt))>1 else None)
            sc_a = StandardScaler().fit(Xa_tr)
            Xa_tr_s = sc_a.transform(Xa_tr).astype(np.float32)
            Xa_te_s = sc_a.transform(Xa_te).astype(np.float32)
            n_neg, n_pos = np.bincount(ya_tr, minlength=2)
            spw_a = n_neg / max(1, n_pos)
            m_a = XGBClassifier(n_estimators=60 if FAST_MODE else 200, learning_rate=0.1, max_depth=6,
                                subsample=0.9, colsample_bytree=0.8, objective="binary:logistic",
                                eval_metric="logloss", tree_method="hist", scale_pos_weight=spw_a,
                                early_stopping_rounds=20, random_state=SEED, n_jobs=-1)
            n_val_a = int(0.15 * len(Xa_tr_s))
            m_a.fit(Xa_tr_s[:-n_val_a], ya_tr[:-n_val_a], eval_set=[(Xa_tr_s[-n_val_a:], ya_tr[-n_val_a:])], verbose=False)
            p_a = m_a.predict_proba(Xa_te_s)[:, 1]
            intra_mcc = matthews_corrcoef(ya_te, (p_a>=0.5).astype(int))
            intra_acc = accuracy_score(ya_te, (p_a>=0.5).astype(int))
            print(f"  INTRA-dataset MCC: {intra_mcc:.4f} | Acc: {intra_acc:.4f}")
            # duplicate rate
            n_alt_dup = int(X_alt.duplicated().sum())
            alt_dup_rate = n_alt_dup / len(X_alt) * 100
            print(f"  duplicate rate: {alt_dup_rate:.2f}%")
            # CROSS-DATASET TRANSFER: use IoTID20-trained XGBoost on alt common features
            if len(common) >= 5 and set(common).issubset(set(FEATURE_COLS_KEEP)):
                common_idx = [FEATURE_COLS_KEEP.index(c) for c in common]
                # train XGBoost on IoTID20 common-feature subset
                xgb_id20_c = XGBClassifier(n_estimators=60 if FAST_MODE else 200, learning_rate=0.1, max_depth=6,
                                            subsample=0.9, colsample_bytree=0.8, objective="binary:logistic",
                                            eval_metric="logloss", tree_method="hist", scale_pos_weight=spw,
                                            early_stopping_rounds=20, random_state=SEED, n_jobs=-1)
                xgb_id20_c.fit(X_train_s[:, common_idx], y_train_np,
                               eval_set=[(X_val_s[:, common_idx], y_val_np)], verbose=False)
                p_id20_c = xgb_id20_c.predict_proba(X_test_s[:, common_idx])[:, 1]
                id20_c_mcc = matthews_corrcoef(y_test_np, (p_id20_c>=0.5).astype(int))
                # transfer: apply to alt dataset (scaled with alt scaler on common features)
                p_transfer = xgb_id20_c.predict_proba(Xa_te_s)[:, 1]
                transfer_mcc = matthews_corrcoef(ya_te, (p_transfer>=0.5).astype(int))
                print(f"  IoTID20 common-feats MCC: {id20_c_mcc:.4f}")
                print(f"  CROSS-dataset transfer MCC: {transfer_mcc:.4f}")
                cross_results.append({"dataset": ds_name, "experiment": "transfer_mcc", "value": transfer_mcc})
                cross_results.append({"dataset": ds_name, "experiment": "id20_common_mcc", "value": id20_c_mcc})
            cross_results.append({"dataset": ds_name, "experiment": "intra_mcc", "value": intra_mcc})
            cross_results.append({"dataset": ds_name, "experiment": "intra_acc", "value": intra_acc})
            cross_results.append({"dataset": ds_name, "experiment": "duplicate_rate_pct", "value": alt_dup_rate})
            del df_alt; gc.collect()
        except Exception as e:
            print(f"  ERROR processing {ds_name}: {e}")
            import traceback; traceback.print_exc()

if cross_results:
    cross_df = pd.DataFrame(cross_results)
    cross_df.to_csv(RESULTS_DIR / "cross_dataset_validation.csv", index=False)
    print("\n=== CROSS-DATASET SUMMARY ===")
    print(cross_df.to_string(index=False))
    # add IoTID20 row for comparison
    iot20_row = pd.DataFrame([
        {"dataset":"IoTID20","experiment":"intra_mcc","value":metrics_df.set_index('Model').loc['XGBoost','MCC']},
        {"dataset":"IoTID20","experiment":"duplicate_rate_pct","value":n_dup/n_rows_raw*100},
    ])
    combined = pd.concat([cross_df, iot20_row], ignore_index=True)
    fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
    mcc_data = combined[combined['experiment']=='intra_mcc']
    ax.bar(mcc_data['dataset'], mcc_data['value'], color=['#0072B2','#D55E00','#009E73','#E69F00'][:len(mcc_data)])
    for i, (_, r) in enumerate(mcc_data.iterrows()):
        ax.text(i, r['value']+0.01, f"{r['value']:.3f}", ha='center')
    ax.set_ylabel("MCC (intra-dataset XGBoost)"); ax.set_title("Cross-dataset intra-dataset performance")
    ax.set_ylim(0, 1.0); ax.tick_params(axis='x', rotation=15)
    finish_fig(fig, "fig19_cross_dataset")
else:
    print("\nNo cross-dataset experiments completed (see notes above).")


## 15 · Statistical significance

McNemar test, bootstrap 95% CI, Wilcoxon signed-rank.

In [ ]:
banner("SECTION 15 - STATISTICAL SIGNIFICANCE")
from scipy.stats import binomtest, wilcoxon

def mcnemar_test(y_true, pred_a, pred_b):
    a_cor = (pred_a == y_true)
    b_cor = (pred_b == y_true)
    b01 = int((a_cor & ~b_cor).sum())
    b10 = int((~a_cor & b_cor).sum())
    n = b01 + b10
    if n == 0: return 0, 1.0, b01, b10
    r = binomtest(b01, n, p=0.5, alternative='two-sided')
    return min(b01, b10), r.pvalue, b01, b10

pairs = [("Logistic Regression","XGBoost"), ("Logistic Regression","Neural Network"), ("XGBoost","Neural Network")]
mc_rows = []
print("--- McNemar ---")
for a, b in pairs:
    st, p, b01, b10 = mcnemar_test(y_test_np, PREDS[a], PREDS[b])
    sig = "***" if p<0.001 else ("**" if p<0.01 else ("*" if p<0.05 else "ns"))
    mc_rows.append({"a":a,"b":b,"b01":b01,"b10":b10,"p":p,"sig":sig})
    print(f"  {a} vs {b}: b01={b01}, b10={b10}, p={p:.4e} {sig}")
pd.DataFrame(mc_rows).to_csv(RESULTS_DIR / "mcnemar_tests.csv", index=False)

print(f"\n--- Bootstrap 95% CI ({BOOTSTRAP_N} resamples) ---")
boot = {n: [] for n in model_names}
rng_b = np.random.RandomState(SEED)
for _ in range(BOOTSTRAP_N):
    idx = rng_b.choice(len(y_test_np), len(y_test_np), replace=True)
    y_b = y_test_np[idx]
    for n in model_names:
        boot[n].append(matthews_corrcoef(y_b, (PROBS[n][idx]>=0.5).astype(int)))
boot_rows = []
for n in model_names:
    arr = np.array(boot[n])
    boot_rows.append({"Model":n,"MCC_mean":float(np.mean(arr)),"MCC_std":float(np.std(arr)),
                      "MCC_ci_low":float(np.percentile(arr,2.5)),"MCC_ci_high":float(np.percentile(arr,97.5))})
    print(f"  {n}: MCC = {np.mean(arr):.4f} +/- {np.std(arr):.4f}, 95% CI [{np.percentile(arr,2.5):.4f}, {np.percentile(arr,97.5):.4f}]")
boot_df = pd.DataFrame(boot_rows)
boot_df.to_csv(RESULTS_DIR / "bootstrap_ci.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
for _, r in boot_df.iterrows():
    c = MODEL_COLORS[r['Model']]
    ax.errorbar([r['Model']], [r['MCC_mean']],
                yerr=[[r['MCC_mean']-r['MCC_ci_low']],[r['MCC_ci_high']-r['MCC_mean']]],
                fmt='o', color=c, capsize=8, markersize=10, lw=2)
ax.set_ylabel("MCC (95% bootstrap CI)"); ax.set_title(f"Bootstrap CIs (n={BOOTSTRAP_N})")
ax.tick_params(axis='x', rotation=15); ax.set_ylim(0, 1.0); ax.grid(True, alpha=0.3)
finish_fig(fig, "fig20_bootstrap_ci")

print("\n--- Wilcoxon ---")
wl_rows = []
for a, b in pairs:
    try:
        st, p = wilcoxon(np.array(boot[a]), np.array(boot[b]))
        sig = "***" if p<0.001 else ("**" if p<0.01 else ("*" if p<0.05 else "ns"))
    except Exception as e:
        st, p, sig = None, None, f"err:{e}"
    wl_rows.append({"a":a,"b":b,"W":st,"p":p,"sig":sig})
    print(f"  {a} vs {b}: W={st}, p={p:.4e} {sig}")
pd.DataFrame(wl_rows).to_csv(RESULTS_DIR / "wilcoxon_tests.csv", index=False)


## 16 · Two-stage classification

Stage 1: binary detector (XGBoost). Stage 2: multi-class attack-type classifier.

In [ ]:
banner("SECTION 16 - TWO-STAGE CLASSIFICATION")
anom_train_mask = (cat_train_np != 'Normal')
anom_test_mask = (cat_test_np != 'Normal')
if anom_train_mask.sum() > 0 and anom_test_mask.sum() > 0:
    attack_cats = sorted(np.unique(cat_train_np[anom_train_mask]))
    cat_to_idx = {c: i for i, c in enumerate(attack_cats)}
    y_mu_tr = np.array([cat_to_idx[c] for c in cat_train_np[anom_train_mask]])
    X_mu_tr = X_train_s[anom_train_mask]
    xgb_mu = XGBClassifier(n_estimators=60 if FAST_MODE else 250, learning_rate=0.1, max_depth=6,
                           subsample=0.9, colsample_bytree=0.8, objective='multi:softprob',
                           num_class=len(attack_cats), eval_metric='mlogloss', tree_method='hist',
                           early_stopping_rounds=20, random_state=SEED, n_jobs=-1)
    n_val_m = int(0.15 * len(X_mu_tr))
    xgb_mu.fit(X_mu_tr[:-n_val_m], y_mu_tr[:-n_val_m],
               eval_set=[(X_mu_tr[-n_val_m:], y_mu_tr[-n_val_m:])], verbose=False)
    flagged = (PREDS['XGBoost'] == 1)
    correct_flag = anom_test_mask & flagged
    if correct_flag.sum() > 0:
        p_mu = xgb_mu.predict(X_test_s[correct_flag])
        y_mu_te = np.array([cat_to_idx[c] for c in cat_test_np[correct_flag]])
        mu_f1 = f1_score(y_mu_te, p_mu, average='macro', zero_division=0)
        mu_acc = accuracy_score(y_mu_te, p_mu)
        print(f"Stage-2 multi-class: acc={mu_acc:.4f}, F1(macro)={mu_f1:.4f}")
        print(classification_report(y_mu_te, p_mu, target_names=attack_cats, zero_division=0))
        cm_mu = confusion_matrix(y_mu_te, p_mu)
        fig, ax = plt.subplots(figsize=(7, 5.5), constrained_layout=True)
        sns.heatmap(cm_mu, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=attack_cats, yticklabels=attack_cats)
        ax.set_xlabel('predicted'); ax.set_ylabel('true')
        ax.set_title(f'Stage-2 confusion (macro F1={mu_f1:.3f})')
        ax.tick_params(axis='x', rotation=30)
        finish_fig(fig, "fig21_two_stage")


## 17 · Ensembles & hard samples

In [ ]:
banner("SECTION 17 - ENSEMBLES & HARD SAMPLES")
vote_maj = (sum(PREDS[n] for n in model_names) >= 2).astype(int)
vote_soft_p = np.mean([PROBS[n] for n in model_names], axis=0)
vote_soft = (vote_soft_p >= 0.5).astype(int)
ens_rows = [
    {"Model":"Majority vote","MCC":matthews_corrcoef(y_test_np, vote_maj),
     "Accuracy":accuracy_score(y_test_np, vote_maj),"PR_AUC":average_precision_score(y_test_np, vote_maj.astype(float))},
    {"Model":"Soft vote","MCC":matthews_corrcoef(y_test_np, vote_soft),
     "Accuracy":accuracy_score(y_test_np, vote_soft),"PR_AUC":average_precision_score(y_test_np, vote_soft_p)},
]
ens_df = pd.DataFrame(ens_rows)
ens_df.to_csv(RESULTS_DIR / "ensembles.csv", index=False)
print(ens_df.round(4).to_string(index=False))
all_wrong = (PREDS['Logistic Regression'] != y_test_np) & (PREDS['XGBoost'] != y_test_np) & (PREDS['Neural Network'] != y_test_np)
print(f"\nHard samples (all 3 wrong): {all_wrong.sum()} / {len(y_test_np)} ({all_wrong.mean()*100:.2f}%)")
if all_wrong.sum() > 0:
    print("Hard-sample categories:")
    print(pd.Series(cat_test_np[all_wrong]).value_counts().to_string())


## 18 · Hard-attacks deep dive (Priority 3)

Focus on **Scan** (F1≈0.62) and **MITM ARP Spoofing** (F1≈0.52) — the two hardest families. We ask:

1. Which features distinguish them from Normal traffic?
2. Which features confuse the model (high SHAP but wrong direction)?
3. How do their feature distributions overlap with Normal?
4. What feature engineering could help?

In [ ]:
banner("SECTION 18 - HARD ATTACKS DEEP DIVE")

# identify the hardest families from per_family_df
hard_families = per_family_df[per_family_df['family'] != 'Normal'].head(2)['family'].tolist()
print(f"Hardest families: {hard_families}")

normal_mask_test = (cat_test_np == 'Normal')
X_test_normal = X_test_s[normal_mask_test]

hard_attack_analysis = []
for fam in hard_families:
    print(f"\n=== {fam} ===")
    fam_mask = (cat_test_np == fam)
    if fam_mask.sum() == 0:
        print(f"  no {fam} samples in test set"); continue
    X_fam = X_test_s[fam_mask]
    print(f"  samples: {len(X_fam)} (Normal: {len(X_test_normal)})")
    
    # feature distribution comparison: mean and std per feature for family vs normal
    fam_mean = X_fam.mean(axis=0)
    nor_mean = X_test_normal.mean(axis=0)
    fam_std = X_fam.std(axis=0)
    nor_std = X_test_normal.std(axis=0)
    
    # separation score: |mean difference| / combined std
    sep_score = np.abs(fam_mean - nor_mean) / (np.sqrt(fam_std**2 + nor_std**2) + 1e-9)
    sep_df = pd.Series(sep_score, index=FEATURE_COLS_KEEP, name='separation_score').sort_values(ascending=False)
    sep_df.to_csv(RESULTS_DIR / f'feature_separation_{fam.replace(" ","_")}.csv')
    
    print(f"  Top 10 MOST separating features (high = easy to distinguish):")
    print(sep_df.head(10).round(3).to_string())
    print(f"  Top 10 LEAST separating features (high overlap with Normal):")
    print(sep_df.tail(10).round(3).to_string())
    
    # SHAP for this family specifically
    if 'shap_arr' in dir() and len(shap_arr) > 0:
        # find which test samples belong to this family (within the shap sample)
        shap_sample_idx = adv_idx[:shap_n] if 'adv_idx' in dir() else np.arange(min(shap_n, len(X_test_s)))
        # actually use the first shap_n test samples
        shap_test_cats = cat_test_np[:shap_n]
        fam_shap_mask = (shap_test_cats == fam)
        if fam_shap_mask.sum() > 0:
            shap_fam = shap_arr[fam_shap_mask]
            shap_fam_importance = np.abs(shap_fam).mean(axis=0)
            shap_fam_df = pd.Series(shap_fam_importance, index=FEATURE_COLS_KEEP, name=f'shap_{fam}').sort_values(ascending=False)
            print(f"  Top 10 SHAP features for {fam}:")
            print(shap_fam_df.head(10).round(4).to_string())
            
            # correlation: separation score vs SHAP importance
            # high SHAP + low separation = confusing feature (model relies on it but it doesn't separate well)
            conf_df = pd.DataFrame({
                'separation': sep_df,
                'shap_importance': shap_fam_importance
            }).fillna(0)
            conf_df['confusion_score'] = conf_df['shap_importance'] / (conf_df['separation'] + 0.01)
            conf_df = conf_df.sort_values('confusion_score', ascending=False)
            print(f"  Top 5 CONFUSING features (high SHAP reliance, low separation):")
            print(conf_df[['separation','shap_importance','confusion_score']].head(5).round(4).to_string())
            hard_attack_analysis.append({
                'family': fam,
                'n_samples': int(fam_mask.sum()),
                'top_separating': sep_df.head(3).index.tolist(),
                'top_confusing': conf_df.head(3).index.tolist(),
            })
    
    # plot distribution comparison for top 3 separating features
    top3 = sep_df.head(3).index.tolist()
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    for ax, feat in zip(axes, top3):
        idx = FEATURE_COLS_KEEP.index(feat)
        vals_nor = X_test_normal[:, idx]
        vals_fam = X_fam[:, idx]
        ax.hist(vals_nor, bins=40, alpha=0.5, color=CAT_COLORS['Normal'], label='Normal', density=True)
        ax.hist(vals_fam, bins=40, alpha=0.5, color=CAT_COLORS.get(fam, '#999'), label=fam, density=True)
        ax.set_xlabel(feat); ax.set_ylabel('density')
        ax.set_title(f'{feat}\nseparation={sep_df[feat]:.2f}')
        ax.legend(fontsize=8)
    fig.suptitle(f'Feature distribution: Normal vs {fam}', fontweight='bold')
    finish_fig(fig, f'fig_hard_attack_{fam.replace(" ","_")}')

if hard_attack_analysis:
    pd.DataFrame(hard_attack_analysis).to_csv(RESULTS_DIR / "hard_attack_analysis.csv", index=False)


## 19 · Multi-seed stability (Priority 1)

Run the full train+eval pipeline with 10 different seeds. Report mean ± std of MCC, F1, PR-AUC for each model. This answers: *is XGBoost really better than LR, or is the difference within seed noise?*

In [ ]:
banner("SECTION 19 - MULTI-SEED STABILITY")
from sklearn.model_selection import train_test_split as _tts

def run_one_seed(seed):
    """Re-split, re-train, re-evaluate all 3 models with the given seed."""
    rng = np.random.RandomState(seed)
    # re-split
    idx_tv, idx_te = _tts(np.arange(len(data)), test_size=TEST_FRAC, random_state=seed, stratify=y_all)
    idx_tr, idx_va = _tts(idx_tv, test_size=VAL_FRAC, random_state=seed, stratify=y_all.iloc[idx_tv])
    X_tr = X_all.iloc[idx_tr].copy()
    X_va = X_all.iloc[idx_va].copy()
    X_te = X_all.iloc[idx_te].copy()
    y_tr = y_all.iloc[idx_tr].to_numpy()
    y_va = y_all.iloc[idx_va].to_numpy()
    y_te = y_all.iloc[idx_te].to_numpy()
    # impute with train median
    med = X_tr.median()
    allnan = med.index[med.isna()].tolist()
    if allnan:
        X_tr = X_tr.drop(columns=allnan)
        X_va = X_va.drop(columns=allnan)
        X_te = X_te.drop(columns=allnan)
    X_tr = X_tr.fillna(med)
    X_va = X_va.fillna(med)
    X_te = X_te.fillna(med)
    # drop zero-variance
    v0 = [c for c in X_tr.columns if X_tr[c].std(ddof=0) == 0]
    if v0:
        X_tr = X_tr.drop(columns=v0)
        X_va = X_va.drop(columns=v0)
        X_te = X_te.drop(columns=v0)
    # scale
    sc = StandardScaler().fit(X_tr)
    X_tr_s = sc.transform(X_tr).astype(np.float32)
    X_va_s = sc.transform(X_va).astype(np.float32)
    X_te_s = sc.transform(X_te).astype(np.float32)
    # class weight
    n_neg, n_pos = np.bincount(y_tr)
    spw_s = n_neg / max(1, n_pos)
    results = {}
    # LR
    m_lr = LogisticRegression(C=1.0, penalty='l2', solver='lbfgs', class_weight='balanced',
                              max_iter=300, random_state=seed)
    m_lr.fit(X_tr_s, y_tr)
    p_lr = m_lr.predict_proba(X_te_s)[:, 1]
    pred_lr = (p_lr >= 0.5).astype(int)
    results['Logistic Regression'] = {
        'MCC': matthews_corrcoef(y_te, pred_lr),
        'F1_macro': f1_score(y_te, pred_lr, average='macro', zero_division=0),
        'PR_AUC': average_precision_score(y_te, p_lr),
    }
    # XGB
    m_xgb = XGBClassifier(n_estimators=60 if FAST_MODE else 200, learning_rate=0.1, max_depth=6,
                          subsample=0.9, colsample_bytree=0.8, objective='binary:logistic',
                          eval_metric='logloss', tree_method='hist', scale_pos_weight=spw_s,
                          early_stopping_rounds=20, random_state=seed, n_jobs=-1)
    m_xgb.fit(X_tr_s, y_tr, eval_set=[(X_va_s, y_va)], verbose=False)
    p_xgb = m_xgb.predict_proba(X_te_s)[:, 1]
    pred_xgb = (p_xgb >= 0.5).astype(int)
    results['XGBoost'] = {
        'MCC': matthews_corrcoef(y_te, pred_xgb),
        'F1_macro': f1_score(y_te, pred_xgb, average='macro', zero_division=0),
        'PR_AUC': average_precision_score(y_te, p_xgb),
    }
    # MLP
    torch.manual_seed(seed)
    net = SmallNet(X_tr_s.shape[1]).to(device)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([spw_s], dtype=torch.float32, device=device))
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5)
    Xtr_t = torch.from_numpy(X_tr_s)
    ytr_t = torch.from_numpy(y_tr.astype(np.float32)).unsqueeze(1)
    Xva_t = torch.from_numpy(X_va_s)
    yva_t = torch.from_numpy(y_va.astype(np.float32)).unsqueeze(1)
    Xte_t = torch.from_numpy(X_te_s)
    ld = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=512, shuffle=True)
    ep_max = 5 if FAST_MODE else 10
    best_v, best_st, wait = np.inf, None, 0
    for ep in range(ep_max):
        net.train()
        for xb, yb in ld:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(net(xb), yb)
            loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            vl = crit(net(Xva_t.to(device)), yva_t.to(device)).item()
        if vl < best_v - 1e-4:
            best_v, wait = vl, 0
            best_st = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        else:
            wait += 1
            if wait >= 2: break
    if best_st is not None:
        net.load_state_dict(best_st)
    net.eval()
    with torch.no_grad():
        p_nn = torch.sigmoid(net(Xte_t.to(device))).cpu().numpy().ravel()
    pred_nn = (p_nn >= 0.5).astype(int)
    results['Neural Network'] = {
        'MCC': matthews_corrcoef(y_te, pred_nn),
        'F1_macro': f1_score(y_te, pred_nn, average='macro', zero_division=0),
        'PR_AUC': average_precision_score(y_te, p_nn),
    }
    return results

print(f"Running {len(MULTI_SEEDS)} seeds: {MULTI_SEEDS}")
multi_rows = []
for i, s in enumerate(MULTI_SEEDS):
    t0 = time.time()
    print(f"\n--- seed {i+1}/{len(MULTI_SEEDS)}: {s} ---")
    try:
        res = run_one_seed(s)
        for model_name, metrics in res.items():
            multi_rows.append({'seed': s, 'Model': model_name, **metrics})
            print(f"  {model_name}: MCC={metrics['MCC']:.4f} F1={metrics['F1_macro']:.4f} PR-AUC={metrics['PR_AUC']:.4f}")
        print(f"  elapsed: {time.time()-t0:.1f}s")
        # cleanup to avoid memory buildup across seeds
        try:
            del m_lr, m_xgb, net
        except NameError:
            pass
        gc.collect()
    except Exception as e:
        print(f"  ERROR: {e}")
        import traceback; traceback.print_exc()

multi_df = pd.DataFrame(multi_rows)
multi_df.to_csv(RESULTS_DIR / "multi_seed_results.csv", index=False)

# summary stats
print("\n=== MULTI-SEED SUMMARY (mean ± std) ===")
summary = multi_df.groupby('Model').agg(
    MCC_mean=('MCC','mean'), MCC_std=('MCC','std'),
    F1_mean=('F1_macro','mean'), F1_std=('F1_macro','std'),
    PR_AUC_mean=('PR_AUC','mean'), PR_AUC_std=('PR_AUC','std'),
    n_seeds=('seed','count')
).round(4)
summary.to_csv(RESULTS_DIR / "multi_seed_summary.csv")
print(summary.to_string())

# plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for metric, ax in zip(['MCC','F1_macro','PR_AUC'], axes):
    for name in model_names:
        sub = multi_df[multi_df['Model']==name]
        ax.plot(sub['seed'], sub[metric], 'o-', color=MODEL_COLORS[name], lw=2, label=name)
    ax.set_xlabel('seed'); ax.set_ylabel(metric)
    ax.set_title(f'{metric} across seeds'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
finish_fig(fig, "fig22_multi_seed_stability")

# box plot
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
data_box = [multi_df[multi_df['Model']==n]['MCC'].values for n in model_names]
bp = ax.boxplot(data_box, tick_labels=model_names, patch_artist=True)
for patch, name in zip(bp['boxes'], model_names):
    patch.set_facecolor(MODEL_COLORS[name]); patch.set_alpha(0.6)
ax.set_ylabel('MCC'); ax.set_title(f'MCC distribution across {len(MULTI_SEEDS)} seeds')
ax.tick_params(axis='x', rotation=15); ax.grid(True, alpha=0.3)
finish_fig(fig, "fig23_multi_seed_boxplot")


## 20 · Final summary

In [ ]:
banner("FINAL SUMMARY")
key_findings = []
key_findings.append(("RAW rows", f"{n_rows_raw:,}"))
key_findings.append(("Unique rows (after de-dup)", f"{len(data):,}"))
key_findings.append(("Duplicate rate (raw)", f"{n_dup/n_rows_raw*100:.2f}%"))
key_findings.append(("Duplicate verification methods agree", str(methods_agree)))
key_findings.append(("Final feature count", str(len(FEATURE_COLS_KEEP))))
for _, r in metrics_df.iterrows():
    key_findings.append((f"{r['Model']} MCC", f"{r['MCC']:.4f}"))
    key_findings.append((f"{r['Model']} PR-AUC", f"{r['PR_AUC']:.4f}"))
# noise methodology comparison
if 'noise_a_df' in dir() and 'noise_b_df' in dir():
    xb_a_01 = noise_a_df[(noise_a_df['Model']=='XGBoost') & (noise_a_df['sigma']==0.1)]['MCC'].values
    xb_b_01 = noise_b_df[(noise_b_df['Model']=='XGBoost') & (noise_b_df['sigma']==0.1)]['MCC'].values
    if len(xb_a_01) and len(xb_b_01):
        key_findings.append(("XGB MCC at sigma=0.1 (raw-feature noise, proper)", f"{xb_a_01[0]:.4f}"))
        key_findings.append(("XGB MCC at sigma=0.1 (scaled-feature noise, original)", f"{xb_b_01[0]:.4f}"))
        key_findings.append(("Methodology artifact detected", str(xb_a_01[0] > xb_b_01[0] + 0.1)))
if 'temporal_mcc' in dir() and temporal_mcc is not None:
    key_findings.append(("XGB temporal-split MCC", f"{temporal_mcc:.4f}"))
    key_findings.append(("Random vs temporal MCC optimism", f"{metrics_df.set_index('Model').loc['XGBoost','MCC'] - temporal_mcc:.4f}"))
if 'multi_df' in dir() and len(multi_df) > 0:
    for name in model_names:
        sub = multi_df[multi_df['Model']==name]
        key_findings.append((f"{name} multi-seed MCC mean±std", f"{sub['MCC'].mean():.4f}±{sub['MCC'].std():.4f} (n={len(sub)})"))
if cross_results:
    for r in cross_results:
        key_findings.append((f"{r['dataset']} - {r['experiment']}", f"{r['value']:.4f}"))
for _, r in boot_df.iterrows():
    key_findings.append((f"{r['Model']} bootstrap 95% CI", f"[{r['MCC_ci_low']:.4f}, {r['MCC_ci_high']:.4f}]"))

with open(RESULTS_DIR / "key_findings.txt", "w") as f:
    f.write("=" * 60 + "\nIoTIDS20-V3 KEY FINDINGS\n" + "=" * 60 + "\n\n")
    for k, v in key_findings:
        f.write(f"{k:<55} : {v}\n")
    f.write("\n" + "=" * 60 + "\n")

print("\n" + "=" * 60)
print("IoTIDS20-V3 KEY FINDINGS")
print("=" * 60 + "\n")
for k, v in key_findings:
    print(f"  {k:<55} : {v}")
print("\n" + "=" * 60)
print(f"Figures: {FIG_DIR} | CSVs: {RESULTS_DIR} | Models: {MODELS_DIR}")
print("=" * 60)
print("\nNotebook complete! Check results/key_findings.txt for the full summary.")
